# SSL Experiment — nb04: Few-Shot Evaluation

**Goal:** Compare SimCLR and CTLS-SSL on N-way K-shot transfer to
CIFAR-100 novel classes, with semantic-distance-stratified analysis.

**Test:** 5-way 1-shot and 5-way 5-shot, 600 episodes each.

**Key hypothesis (from CTLS-SSL theory):**
CTLS-SSL should outperform SimCLR more for *close* novel classes (small_mammals,
vehicles_1 — semantically similar to CIFAR-10) than for *distant* novel classes
(trees — no CIFAR-10 counterpart). If the advantage is uniform across groups,
the circuit scaffold is not the cause of improvement.

**Novel classes (CIFAR-100 superclasses 15–19):**
- Superclass 15 — reptiles (medium): crocodile, dinosaur, lizard, snake, turtle
- Superclass 16 — small_mammals (close): hamster, mouse, rabbit, shrew, squirrel
- Superclass 17 — trees (distant): maple_tree, oak_tree, palm_tree, pine_tree, willow_tree
- Superclass 18 — vehicles_1 (close): bicycle, bus, motorcycle, pickup_truck, train
- Superclass 19 — vehicles_2 (medium): lawn_mower, rocket, streetcar, tank, tractor

## 0. Setup

In [ ]:
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = 'https://github.com/jacobposchl/model-interpretability'
    REPO_DIR = '/content/ctls'

    if not os.path.exists(REPO_DIR):
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
        os.system(f"pip install -r {REPO_DIR}/requirements.txt -q")

    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_BASE = '/content/drive/MyDrive/ctls'
    os.makedirs(f"{DRIVE_BASE}/data/raw", exist_ok=True)
    os.makedirs(f"{DRIVE_BASE}/experiments", exist_ok=True)

    for link, target in [
        (f"{REPO_DIR}/data/raw",    f"{DRIVE_BASE}/data/raw"),
        (f"{REPO_DIR}/experiments", f"{DRIVE_BASE}/experiments"),
    ]:
        if os.path.islink(link): os.unlink(link)
        os.symlink(target, link)

    ROOT = REPO_DIR
else:
    ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print(f"Working directory: {os.getcwd()} ({'Colab' if IN_COLAB else 'local'})")

In [ ]:
import yaml, torch
import numpy as np
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open('configs/ssl/simclr.yaml') as f:
    cfg_simclr = yaml.safe_load(f)
with open('configs/ssl/ctls_ssl.yaml') as f:
    cfg_ctls = yaml.safe_load(f)

DATA_DIR = cfg_simclr['data']['data_dir']

## 1. Load Models

In [ ]:
from models.soft_mask import SoftMask
from models.backbone import CTLSBackbone
from models.meta_encoder import MetaEncoder

def load_model_from_ckpt(config, ckpt_path, device):
    mcfg = config['model']; ecfg = mcfg['meta_encoder']
    smcfg = mcfg.get('soft_mask', {})
    soft_mask = SoftMask(init_temperature=smcfg.get('init_temperature', 1.0))
    backbone = CTLSBackbone(
        arch=mcfg['arch'], num_classes=mcfg['num_classes'],
        soft_mask=soft_mask, pretrained=False,
    ).to(device)
    meta_encoder = MetaEncoder(
        layer_dims=backbone.layer_dims,
        hidden_dim=ecfg.get('hidden_dim', 256),
        embedding_dim=ecfg.get('embedding_dim', 64),
        encoder_type=ecfg.get('encoder_type', 'mlp'),
    ).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    backbone.load_state_dict(ckpt['backbone_state'])
    meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
    backbone.eval(); meta_encoder.eval()
    return backbone, meta_encoder

backbone_simclr, meta_simclr = load_model_from_ckpt(cfg_simclr, 'experiments/ssl/simclr/best.pt', DEVICE)
backbone_ctls,   meta_ctls   = load_model_from_ckpt(cfg_ctls,   'experiments/ssl/ctls_ssl/best.pt', DEVICE)

print('Models loaded.')


## 2. Build CIFAR-100 Novel Class Dataset

In [ ]:
from data.ssl import CIFAR100ForTransfer
from data.cifar import get_val_transform

novel_dataset = CIFAR100ForTransfer(
    root=DATA_DIR,
    train=False,          # use test split for evaluation
    transform=get_val_transform(),
    split='novel',
    download=True,
)
print(f"Novel dataset size: {len(novel_dataset)} samples")
print(f"Novel classes ({len(novel_dataset.class_to_indices)}): {sorted(novel_dataset.class_to_indices.keys())}")

## 3. Few-Shot Evaluation

5-way 1-shot and 5-way 5-shot, 600 episodes each, for SimCLR and CTLS-SSL.

In [ ]:
from evaluation.fewshot import EpisodeSampler, FewShotEvaluator

N_EPISODES = 600

results = {}

for name, (backbone, meta_encoder) in [
    ('SimCLR', (backbone_simclr, meta_simclr)),
    ('CTLS-SSL', (backbone_ctls, meta_ctls)),
]:
    evaluator = FewShotEvaluator(backbone, meta_encoder, DEVICE)
    results[name] = {}

    for k_shot in [1, 5]:
        print(f"Evaluating {name} 5-way {k_shot}-shot...")
        sampler = EpisodeSampler(
            novel_dataset, n_way=5, k_shot=k_shot, n_query=15, n_episodes=N_EPISODES
        )
        r = evaluator.evaluate(sampler)
        results[name][k_shot] = r
        print(f"  5-way {k_shot}-shot: {r['mean_acc']:.4f} ± {r['ci95']:.4f}")

## 4. Results Table

In [ ]:
print('\n=== 5-way K-shot Few-Shot Accuracy (CIFAR-100 novel classes) ===')
print(f"{'Model':<12} {'1-shot acc':>12} {'1-shot CI':>10} {'5-shot acc':>12} {'5-shot CI':>10}")
print('-' * 58)
for name in ['SimCLR', 'CTLS-SSL']:
    r1 = results[name][1]
    r5 = results[name][5]
    print(f"{name:<12} {r1['mean_acc']:>12.4f} {r1['ci95']:>10.4f} {r5['mean_acc']:>12.4f} {r5['ci95']:>10.4f}")

# Delta CTLS-SSL vs SimCLR
print('\n--- Delta: CTLS-SSL − SimCLR ---')
for k in [1, 5]:
    delta = results['CTLS-SSL'][k]['mean_acc'] - results['SimCLR'][k]['mean_acc']
    print(f"  5-way {k}-shot delta: {delta:+.4f}")

## 5. Semantic-Distance-Stratified Analysis

The main hypothesis test: CTLS-SSL advantage should be largest for
**close** novel classes (small_mammals, vehicles_1) and smallest for **distant**
novel classes (trees). If the pattern holds, it validates the circuit scaffold
mechanism.

In [ ]:
from evaluation.fewshot import SemanticDistanceGrouper

grouper = SemanticDistanceGrouper()

# Per-class few-shot evaluation (1-shot): evaluate on each novel class separately
# by running episodes restricted to that class vs. 4 random other novel classes

def per_class_fewshot(backbone, meta_encoder, novel_dataset, n_episodes=100):
    """Estimate per-class 5-way 1-shot accuracy for each novel class."""
    from collections import defaultdict
    evaluator = FewShotEvaluator(backbone, meta_encoder, DEVICE)

    novel_classes = sorted(novel_dataset.class_to_indices.keys())
    per_class_correct = defaultdict(list)

    import random
    for _ in range(n_episodes):
        # Sample episode with 5-way 1-shot
        sampler = EpisodeSampler(novel_dataset, n_way=5, k_shot=1, n_query=15, n_episodes=1)
        for s_imgs, s_labels, q_imgs, q_labels in sampler:
            # Map episode labels back to fine labels for grouping
            # (We can't directly, but we can compute per-episode accuracy)
            pass

    # Simpler approach: run 5-way 1-shot restricted to each superclass pair
    from data.ssl import _CIFAR100_FINE_TO_COARSE
    superclass_to_fines = defaultdict(list)
    for fine in novel_classes:
        sc = _CIFAR100_FINE_TO_COARSE[fine]
        superclass_to_fines[sc].append(fine)

    per_class_acc = {}
    # For each superclass that has enough fine classes, run within-superclass few-shot
    # and attribute accuracy to its fine labels
    for sc, fines in superclass_to_fines.items():
        if len(fines) < 2:
            continue
        # Build a mini-dataset for this superclass
        from data.ssl import CIFAR100ForTransfer
        from torch.utils.data import Subset

        # Get indices for this superclass only
        sc_indices = []
        sc_class_map = defaultdict(list)
        for pos, (fine_label) in enumerate(novel_dataset._labels):
            if _CIFAR100_FINE_TO_COARSE[fine_label] == sc:
                sc_indices.append(pos)
                sc_class_map[fine_label].append(len(sc_indices) - 1)

        # Create a proxy dataset
        class SubsetWithIndex:
            def __init__(self, parent, indices, cls_map):
                self.parent = parent
                self.indices = indices
                self.class_to_indices = cls_map
            def __len__(self):
                return len(self.indices)
            def __getitem__(self, idx):
                return self.parent[self.indices[idx]]

        sc_ds = SubsetWithIndex(novel_dataset, sc_indices, dict(sc_class_map))
        if len(sc_ds.class_to_indices) < 2:
            continue

        n_way_local = min(len(sc_ds.class_to_indices), 5)
        try:
            sc_sampler = EpisodeSampler(sc_ds, n_way=n_way_local, k_shot=1,
                                         n_query=10, n_episodes=50)
            sc_result = evaluator.evaluate(sc_sampler)
            for fine in fines:
                per_class_acc[fine] = sc_result['mean_acc']
        except Exception as e:
            print(f"Skipped superclass {sc}: {e}")

    return per_class_acc

print("Computing per-class accuracies for semantic stratification...")
per_class_simclr = per_class_fewshot(backbone_simclr, meta_simclr, novel_dataset)
per_class_ctls   = per_class_fewshot(backbone_ctls,   meta_ctls,   novel_dataset)
print("Done.")

In [ ]:
# Group by semantic distance and compute mean per group
grouped_simclr = grouper.group_results(per_class_simclr)
grouped_ctls   = grouper.group_results(per_class_ctls)

groups = ['close', 'medium', 'distant']
x = np.arange(len(groups))
width = 0.3

fig, ax = plt.subplots(figsize=(9, 5))
bars_sim = ax.bar(x - width/2, [grouped_simclr.get(g, 0) for g in groups],
                   width, label='SimCLR', color='steelblue', alpha=0.8)
bars_ctls = ax.bar(x + width/2, [grouped_ctls.get(g, 0) for g in groups],
                    width, label='CTLS-SSL', color='darkorange', alpha=0.8)

# Annotate delta
for i, g in enumerate(groups):
    delta = grouped_ctls.get(g, 0) - grouped_simclr.get(g, 0)
    ax.text(i, max(grouped_simclr.get(g, 0), grouped_ctls.get(g, 0)) + 0.005,
            f'{delta:+.3f}', ha='center', fontsize=9, color='black')

ax.set_xticks(x)
ax.set_xticklabels(['Close\n(small_mammals,\nvehicles_1)',
                     'Medium\n(reptiles,\nvehicles_2)',
                     'Distant\n(trees)'])
ax.set_ylabel('5-way 1-shot accuracy')
ax.set_title('CTLS-SSL vs SimCLR: per-semantic-group few-shot accuracy')
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
os.makedirs('experiments/ssl', exist_ok=True)
fig.savefig('experiments/ssl/fewshot_semantic_groups.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nSemantic group accuracy comparison:')
print(f"{'Group':<10} {'SimCLR':>10} {'CTLS-SSL':>10} {'Delta':>10}")
print('-' * 42)
for g in groups:
    sim = grouped_simclr.get(g, float('nan'))
    ctls = grouped_ctls.get(g, float('nan'))
    print(f"{g:<10} {sim:>10.4f} {ctls:>10.4f} {ctls-sim:>+10.4f}")

## 6. Hypothesis Assessment

The CTLS-SSL circuit scaffold hypothesis predicts:
- **Close group delta > Medium group delta > Distant group delta**

Fill in the observed pattern and assess:

| Group | Predicted rank | Observed CTLS-SSL − SimCLR | Consistent? |
|-------|---------------|---------------------------|-------------|
| Close (small_mammals, vehicles_1) | Largest advantage | ___ | ___ |
| Medium (reptiles, vehicles_2) | Medium advantage | ___ | ___ |
| Distant (trees) | Smallest / no advantage | ___ | ___ |

**Next:** Run `nb05_circuit_analysis.ipynb` for circuit-specific diagnostics.